<a href="https://colab.research.google.com/github/apeonqwerty/mnist-handwritten-digit-classification/blob/main/MNIST_Handwritten_Digit_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### 0. Importing Libraries

In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision.datasets as datasets
from torchvision.transforms import ToTensor

#### 1. Load MNIST dataset

In [2]:
mnist_train = datasets.MNIST(root='./data', download=True, train=True, transform=ToTensor())
mnist_test = datasets.MNIST(root='./data', download=True, train=False, transform=ToTensor())

100%|██████████| 9.91M/9.91M [00:00<00:00, 42.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.19MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.6MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.98MB/s]


In [3]:
train_dataloader = DataLoader(mnist_train, batch_size=32, shuffle=True)
test_dataloader = DataLoader(mnist_test, batch_size=32, shuffle=False)

#### 2. Defining the model, the loss function and the optimizer

In [5]:
model = nn.Sequential(
    nn.Flatten(),  # Automatically flatten the input images to a vector
    nn.Linear(784, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, 10)
)

In [6]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [8]:
# If GPU is available, move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=50, bias=True)
  (4): ReLU()
  (5): Linear(in_features=50, out_features=10, bias=True)
)

#### 3. Training

In [9]:
for epoch in range(10):
    model.train()
    loss_sum = 0
    for X, y in train_dataloader:
        X, y = X.to(device), y.to(device)  # Move data to device
        optimizer.zero_grad()
        outputs = model(X)
        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    print(f"Epoch {epoch+1}, Training Loss: {loss_sum / len(train_dataloader)}")

Epoch 1, Training Loss: 0.30140409714927274
Epoch 2, Training Loss: 0.128961135802418
Epoch 3, Training Loss: 0.08840293039362877
Epoch 4, Training Loss: 0.06798235686620077
Epoch 5, Training Loss: 0.0547713665436022
Epoch 6, Training Loss: 0.04413950396712559
Epoch 7, Training Loss: 0.03628684067616705
Epoch 8, Training Loss: 0.030339236012345644
Epoch 9, Training Loss: 0.026069270264562995
Epoch 10, Training Loss: 0.02404372348442533


#### 4. Evaluation

In [10]:
model.eval()
with torch.inference_mode():
    accurate = 0
    total = 0
    loss_sum = 0
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)  # Move data to device
        outputs = model(X)

        # Calculate loss
        loss = loss_fn(outputs, y)
        loss_sum += loss.item()

        # Calculate accuracy
        correct_pred = (y == torch.argmax(outputs, dim=1))
        total += y.size(0)
        accurate += correct_pred.sum().item()

    avg_test_loss = loss_sum / len(test_dataloader)
    accuracy = (accurate / total) * 100
    print(f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

Test Loss: 0.0902, Test Accuracy: 97.59%
